# CARACTERIZAÇÃO DO PROJECTO DE CAMPANHAS

## Análise Completa da Base de Clientes e Plano de Campanhas

Este notebook caracteriza:
1. **Visão geral do projecto** – distribuição temporal e por tipo de oferta
2. **Campanhas de hoje** – volumetria e escassez
3. **Escassez de todas as campanhas** – nível crítico/alto/médio/baixo
4. **Base de clientes por Tier** – antiguidade da última campanha recebida
5. **Base de clientes por Pressão Comercial** – n.º de campanhas recebidas
6. **Análise cruzada Tier × Pressão Comercial**
7. **Elegibilidade e cobertura** – % da base elegível por campanha

> **Pré-requisito:** ligação ao SQL Server (Diomedes / tempdb) e ficheiro `plano.csv`.

In [ ]:
# =============================================================================
# BLOCO 1: CONFIGURAÇÃO E CARREGAMENTO DOS DADOS
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import pyodbc

# ---------------------------------------------------------------------------
# Configurações gerais
# ---------------------------------------------------------------------------
PATH_PLANO     = r'C:\Users\COTAGO\Desktop\Base\plano.csv'
SEPARADOR      = ';'
ENCODING       = 'utf-8-sig'

# Thresholds de escassez (consistentes com o motor de optimização)
ESCASSEZ_THRESHOLD_CRITICA = 0.80
ESCASSEZ_THRESHOLD_ALTA    = 0.50
ESCASSEZ_THRESHOLD_MEDIA   = 0.30

# Data de referência
hoje      = datetime.now()
hoje_str  = hoje.strftime('%Y-%m-%d')
mes_atual = int(hoje.strftime('%Y%m'))

print("=" * 80)
print("  CARACTERIZAÇÃO DO PROJECTO DE CAMPANHAS")
print(f"  Executado em: {hoje.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print()

# ---------------------------------------------------------------------------
# Ligação ao SQL Server
# ---------------------------------------------------------------------------
conn_str = (
    'Driver={SQL Server};'
    'Server=Diomedes;'
    'Database=tempdb;'
    'trusted_connection=yes;'
)
conn   = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("✓ Ligação ao SQL Server estabelecida.")


In [ ]:
# =============================================================================
# BLOCO 2: QUERY #BaseEnvio → df_baseenvio
# =============================================================================
# Executa a query completa que constrói a tabela temporária #BaseEnvio e
# carrega o resultado para o DataFrame df_baseenvio.
# =============================================================================

query_base_envio = """
/* =========================================================
   QUERY SIMPLIFICADA - CAMPANHAS DE ELEGIBILIDADE
   Lógica reorganizada para melhor clareza e manutenção
========================================================= */

-- 1. TABELAS TEMPORÁRIAS PARA GESTÃO DE DEPENDÊNCIAS
DROP TABLE IF EXISTS #temp_tables;
DECLARE @temp_tables TABLE (table_name NVARCHAR(100));
INSERT INTO @temp_tables VALUES
('#tab_max_dt_eleg'),('#SemSolvabilidade'),('#proposta_comercial_CCR'),('#UniversoAux'),('#CRC5G'),
('#Consolida'),('#Reembolsos'),('#UniversoCCR'),('#Obra01'),('#TabelaFinal'),('#TabelaFinalCCR'),
('#Cruzamento'),('#Universo'),('#BaseConsolidada'),('#BaseFinal'),('#Resultado_Financeiro'),
('#BaseEnvio'),('#PMS'),('#proposta_comercial_RUC'),('#proposta_comercial_PCD_Intermedios'),
('#proposta_comercial_PCD'),('#proposta_comercial_PCDAuto'),('#proposta_comercial_PMS'),('#ElegiveisPMS'), ('#tab_max_dt_CRC');

DECLARE @table_name NVARCHAR(100);
DECLARE drop_cursor CURSOR FOR SELECT table_name FROM @temp_tables;
OPEN drop_cursor;
FETCH NEXT FROM drop_cursor INTO @table_name;
WHILE @@FETCH_STATUS = 0
BEGIN
    EXEC('DROP TABLE IF EXISTS ' + @table_name);
    FETCH NEXT FROM drop_cursor INTO @table_name;
END
CLOSE drop_cursor;
DEALLOCATE drop_cursor;

/* =========================================================
   1.1 DATA MÁXIMA DE ELEGIBILIDADE
========================================================= */
SELECT MAX(IdDia) AS max_dt
INTO #tab_max_dt_eleg
FROM armada.dbo.DM_CampanhasElegibilidade;


/* =========================================================
   1.1 DATA MÁXIMA DE ELEGIBILIDADE
========================================================= */
SELECT MAX(DtUltCRC) AS max_dtCRC
INTO #tab_max_dt_CRC
FROM armada.dbo.DM_CampanhasElegibilidade;

/* =========================================================
   2. CLIENTES SEM DADOS DE SOLVABILIDADE
========================================================= */
SELECT DISTINCT 
    ce.NIF,
    CASE 
        WHEN ce.Rendimento IS NULL OR ce.DespesasSolv IS NULL THEN 100 
        ELSE ce.MensMaxSolvabilidade 
    END AS MensMaxSolvabilidadeNEW,
    0 AS excl_solvNEW,
    0 as excl_rendNEW,
    0 as excl_DONew
INTO #SemSolvabilidade
FROM armada.dbo.DM_CampanhasElegibilidade ce
INNER JOIN #tab_max_dt_eleg e ON ce.IdDia = e.max_dt
CROSS JOIN #tab_max_dt_CRC c
WHERE ce.Rendimento IS NULL 
   OR ce.DespesasSolv IS NULL 
   OR ce.TxSolvabilidade IS NULL
   or ce.DtUltCRC IS NULL 
   OR (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 6, CONVERT(DATE, CAST(ce.DtUltCRC AS CHAR(8)))), 'yyyyMMdd')))


/* =========================================================
   3. APURAMENTO CCR - FUNÇÕES MODULARIZADAS
========================================================= */

-- 3.1. TARIFICAÇÃO
SELECT
    a.AMOUNT AS Montante,
    a.tan,
    a.TAEG,
    a.DEADLINE AS Prazo,
    a.MONTHLYPAYWITHINSURANCE AS MM,
    a.MONTHLYPAYWITHINSURANCE - a.MONTHLYPAYWITHOUTINSURANCE AS Seguro,
    a.MTIC
INTO #proposta_comercial_CCR
FROM armada.dbo.ST_DM_CondicoesSimulacao a
JOIN armada.dbo.DM_Oferta b ON a.idoffer = b.idOferta
JOIN armada.dbo.IDH_Dim_Produto c ON b.Produto = c.CodProdutoDAH
WHERE c.CodProduto = 'CCD'
  AND a.AMOUNT IN (5000,6000,7000,8000,9000,10000,11000,12000,13000,14000,
                   15000,16000,17000,18000,19000,20000,21000,22000,23000,24000,
                   25000,26000,27000,28000,29000,30000,35000,40000,45000,50000)
  AND a.DEADLINEWITHINSURANCE = 84;

CREATE CLUSTERED INDEX IX_Tarificacao ON #proposta_comercial_CCR (Montante);

-- 3.2. UNIVERSO BASE FILTRADO
SELECT
a1.NIF,
CASE
WHEN a1.AtividadeCredito = 'Terminado' THEN 'Terminados'
ELSE 'SegEmCurso'
END AS Segmento,
COALESCE(a1.DividaRevolving,0) + COALESCE(a1.DividaCartao,0) +
COALESCE(a1.DividaClassico,0) + COALESCE(a1.DividaAuto,0) AS DividaCRC,
a1.PrestMes,
a1.MesesTerminoMinimo
INTO #UniversoAux
FROM armada.dbo.DM_CampanhasElegibilidade a1
LEFT JOIN #SemSolvabilidade s ON s.NIF = a1.NIF
INNER JOIN #tab_max_dt_eleg e ON a1.IdDia = e.max_dt
CROSS JOIN #tab_max_dt_CRC c
WHERE a1.Ordem = 1 and 
NOT (
a1.excl_financiavel = 1
OR a1.excl_nacionalidade = 1
OR a1.excl_PNM = 1
OR a1.excl_Impagos = 1
OR a1.excl_TipoResidencia = 1
OR a1.excl_DSTI = 1
OR a1.excl_idade = 1
OR a1.excl_RevolvingAtivo = 1
OR a1.excl_UltAbert_Conso = 1
OR a1.excl_PRD = 1
OR a1.excl_removidos = 1
OR a1.excl_IncidentesBancarios = 1
OR a1.excl_RequerInvestigacao = 1
OR a1.excl_prof_instavel = 1
)
AND a1.MesesUltAbertura >= 4 AND (s.excl_solvNEW = 0 or excl_solv = 0) and (excl_DO = 0 or excl_DONew = 0)
 
CREATE CLUSTERED INDEX IX_UniversoAux ON #UniversoAux (NIF);

-- 3.3. CRC 5G
DROP TABLE IF EXISTS #CRC5G
-- Verificar e excluir valores não numéricos da coluna idEnt
SELECT DISTINCT *
INTO #CRC5G
FROM (
    SELECT
        a1.idEnt,
        SUM(CASE
            WHEN LEFT(a1.dissTpInst, 3) IN ('005', '004') THEN 
                COALESCE(TRY_CAST(a1.dissMontTotal AS FLOAT), 0) * 0.03
            WHEN pr.PrazoResidual > 0 THEN 
                COALESCE(TRY_CAST(a1.dissMontTotal AS FLOAT), 0) / pr.PrazoResidual
            ELSE 0
        END) AS MM_Consolida,
        SUM(CASE
            WHEN pr.PrazoResidual > 0 AND a1.dissTpInst IN ('0130', '0140') THEN 
                COALESCE(TRY_CAST(a1.dissMontTotal AS FLOAT), 0) / pr.PrazoResidual
            ELSE 0
        END) AS MM_Consolida_Classicos,
        COUNT(CASE
            WHEN TRY_CAST(a1.dissMontTotal AS FLOAT) > 0 THEN 1
            END) AS n_creditos
    FROM armada.dbo.CRC_DM_Disseminacao5G a1 WITH (NOLOCK)
    INNER JOIN #UniversoAux a ON a.NIF = a1.idEnt
    INNER JOIN (
        SELECT idEnt, MAX(dtInsercao) AS max_dtInsercao
        FROM armada.dbo.CRC_DM_Disseminacao5G
        WHERE ISNUMERIC(idEnt) = 1           -- EXCLUI VALORES NÃO NUMÉRICOS EM idEnt
        GROUP BY idEnt
    ) a2 ON a1.idEnt = a2.idEnt AND a1.dtInsercao = a2.max_dtInsercao
    CROSS APPLY (
        SELECT CASE
            WHEN TRY_CONVERT(DATE, a1.dissDtMatInst) IS NOT NULL
                 AND TRY_CONVERT(DATE, a1.dissDtMatInst) <> '9999-12-31'
            THEN DATEDIFF(MONTH, a1.dtRefDiss, TRY_CONVERT(DATE, a1.dissDtMatInst))
        END AS PrazoResidual
    ) pr
    WHERE ISNUMERIC(a1.idEnt) = 1               -- FILTRA idEnt APENAS NUMÉRICO
      AND (a1.dissTpInst IN ('0130', '0140') OR LEFT(a1.dissTpInst, 3) IN ('005', '004'))
      AND a1.dissTpResp IN ('001', '002')
      AND a1.dissNumDev = '1'
      AND ISNUMERIC(a1.dissMontTotal) = 1      -- FILTRA dissMontTotal PARA VALORES NUMÉRICOS
    GROUP BY a1.idEnt
) AS t
WHERE n_creditos > 1;

CREATE CLUSTERED INDEX IX_CRC5G ON #CRC5G (idEnt);

-- 3.4. CONSOLIDAÇÃO
SELECT
    a1.NIF,
    SUM(CASE WHEN IdProdutoFinanceiro IN (2,9,11,12) THEN 1 ELSE 0 END) AS N_Consolida,
    SUM(CASE WHEN IdProdutoFinanceiro IN (11,12) THEN 1 ELSE 0 END) AS N_ClassicoAuto,
    SUM(CASE WHEN IdProdutoFinanceiro IN (2,9,11,12) 
             THEN TRY_CAST(ValorAgregSaldo AS FLOAT) ELSE 0 END) AS DividaCRC
INTO #Consolida
FROM armada.dbo.IDH_DM_BPNovaCRC a1
INNER JOIN #UniversoAux a ON a.NIF = a1.NIF
WHERE IdSituacaoCredito = 1
  AND IdNivelResp = 1
GROUP BY a1.NIF;

CREATE CLUSTERED INDEX IX_Consolida ON #Consolida (NIF);

-- 3.5. REEMBOLSOS
SELECT
    a3.NIF,
    SUM(TRY_CAST(a1.MntReembolso AS FLOAT)) AS MntReembolso
INTO #Reembolsos
FROM armada.dbo.DM_Reembolsos a1
JOIN armada.dbo.DM_ClienteProcesso a3 ON a1.NumDossier = a3.NumDossier AND a3.Ordem = 1
WHERE a1.TipoReembolso = 'Total'
  AND TRY_CAST(a1.MntReembolso AS FLOAT) > 0
  AND a3.NIF <> 0
  AND LEFT(TRY_CAST(a1.NumDossier AS VARCHAR(20)),1) = '3'
GROUP BY a3.NIF;

CREATE CLUSTERED INDEX IX_Reembolsos ON #Reembolsos (NIF);

-- 3.6. UNIVERSO CCR FINAL
SELECT
    u.NIF,
    u.MesesTerminoMinimo,
    u.Segmento,
    CASE
        WHEN u.Segmento <> 'Terminados' THEN c.DividaCRC
        WHEN u.Segmento = 'Terminados' AND u.MesesTerminoMinimo > 12 THEN r.MntReembolso
        ELSE u.DividaCRC
    END AS DividaCRC,
    u.PrestMes,
    CASE WHEN c.N_Consolida > 1 THEN 1 ELSE 0 END AS MaisDoQue1CreditoConsolida,
    CASE WHEN c.N_ClassicoAuto = c.N_Consolida AND c.N_Consolida > 0 THEN 1 ELSE 0 END AS iSoAutoClassico,
    g.MM_Consolida,
    g.MM_Consolida_Classicos,
    g.n_creditos
INTO #UniversoCCR
FROM #UniversoAux u
LEFT JOIN #Consolida c ON u.NIF = c.NIF AND u.Segmento <> 'Terminados'
LEFT JOIN #Reembolsos r ON u.NIF = r.NIF AND u.Segmento = 'Terminados' AND u.MesesTerminoMinimo > 12
LEFT JOIN #CRC5G g ON TRY_CAST(u.NIF AS VARCHAR(50)) = TRY_CAST(g.idEnt AS VARCHAR(50))
WHERE (u.Segmento <> 'Terminados' AND c.N_Consolida > 1)
   OR (u.Segmento = 'Terminados' AND u.MesesTerminoMinimo <= 12 AND u.DividaCRC > 10000)
   OR (u.Segmento = 'Terminados' AND u.MesesTerminoMinimo > 12 AND r.MntReembolso > 10000);

CREATE CLUSTERED INDEX IX_UniversoCCR ON #UniversoCCR (NIF);

-- 3.7. CÁLCULO SOLVABILIDADE + DSTI
SELECT
    u.*,
    e.Ordem,
    CASE
        WHEN e.RendSolv IS NOT NULL AND e.DespesasSolv IS NOT NULL
        THEN TRY_CAST(e.RendSolv AS FLOAT) - 
             (TRY_CAST(e.DespesasSolv AS FLOAT) - COALESCE(u.MM_Consolida,0))
        ELSE 0
    END AS MensMaxSolvabilidade,
    CASE
        WHEN e.RendSolv IS NOT NULL AND e.PrestMes IS NOT NULL
        THEN TRY_CAST(e.RendSolv AS FLOAT) - 
             (TRY_CAST(e.PrestMes AS FLOAT) - COALESCE(u.MM_Consolida_Classicos,0))
        ELSE 0
    END AS MensMaxDSTI
INTO #Obra01
FROM #UniversoCCR u
INNER JOIN armada.dbo.DM_CampanhasElegibilidade e ON u.NIF = e.NIF
INNER JOIN #tab_max_dt_eleg eleg ON eleg.max_dt = e.IdDia
INNER JOIN #UniversoAux aux ON aux.NIF = e.NIF;

CREATE CLUSTERED INDEX IX_Obra01 ON #Obra01 (NIF);

DROP TABLE IF EXISTS #TempNIFs;

SELECT 
NIF,
LEFT(CodPostal, 2) AS CodPostalPrefix
INTO #TempNIFs
FROM armada.dbo.DM_CampanhasElegibilidade_NIF;

-- Realizar o JOIN com a tabela temporária, usando o prefixo calculado
DROP TABLE IF EXISTS #Lojas

SELECT DISTINCT 
n.NIF, 
COALESCE(l.Loja, 'LISBOA') AS Loja
INTO #Lojas
FROM #TempNIFs n
INNER JOIN DataExperimental_DICV.dbo.Lojas l 
ON l.CodPostalMini = n.CodPostalPrefix;



-- 3.8. TABELA FINAL CCR
SELECT
    o.NIF,
    o.Ordem,
    o.Segmento,
    l.Loja,
    o.iSoAutoClassico,
    o.DividaCRC,
    o.PrestMes,
    o.MM_Consolida,
    o.MM_Consolida_Classicos,
    p.SegSMS,
    p.Montante,
    p.MontanteMax,
    p.Prazo,
    p.PrazoMax,
    p.TAN,
    p.TANMax,
    p.TAEG,
    p.TAEGMax,
    p.MM,
    p.MMMax,
    p.Seguro,
    p.SeguroMax,
    p.MTIC,
    p.MTICMax
INTO #TabelaFinal
FROM #Obra01 o
LEFT JOIN #Lojas l on l.NIF = o.NIF
OUTER APPLY (
    SELECT TOP 1
        CASE
            WHEN o.Segmento <> 'Terminados' AND o.iSoAutoClassico = 1 
                 AND o.MM_Consolida - t.MM > 10 THEN 'SMSA'
            WHEN o.Segmento <> 'Terminados' THEN 'SMSB'
            WHEN o.Segmento = 'Terminados' AND o.MesesTerminoMinimo <= 12 THEN 'SMSC'
            WHEN o.Segmento = 'Terminados' AND o.MesesTerminoMinimo > 12 THEN 'SMSD'
        END AS SegSMS,
        t.Montante,
        t.Prazo,
        t.TAN,
        t.TAEG,
        t.MM,
        t.Seguro,
        t.MTIC,
        t.Montante AS MontanteMax,
        t.Prazo AS PrazoMax,
        t.TAN AS TANMax,
        t.TAEG AS TAEGMax,
        t.MM AS MMMax,
        t.Seguro AS SeguroMax,
        t.MTIC AS MTICMax
    FROM #proposta_comercial_CCR t
    WHERE o.DividaCRC > 500
      AND (
          (o.Segmento <> 'Terminados' AND o.MensMaxSolvabilidade > t.MM
           AND o.MensMaxDSTI > t.MM AND t.Montante > o.DividaCRC + 1000)
          OR (o.Segmento = 'Terminados' AND o.MesesTerminoMinimo <= 12 
              AND t.Montante > o.DividaCRC + 1000)
          OR (o.Segmento = 'Terminados' AND o.MesesTerminoMinimo > 12 
              AND t.Montante > o.DividaCRC)
      )
    ORDER BY CASE
        WHEN o.Segmento <> 'Terminados' AND o.iSoAutoClassico = 1 
        THEN t.Montante - (o.DividaCRC + 1000)
        ELSE t.Montante - o.DividaCRC
    END ASC
) p
WHERE p.Montante IS NOT NULL;

-- 3.9. RESULTADO FINAL CCR
SELECT 
    a.*,
    CASE WHEN SegSMS LIKE 'SMSA' THEN 1 ELSE 0 END AS iXS_CCR_SMSA,
    CASE WHEN SegSMS LIKE 'SMSB' THEN 1 ELSE 0 END AS iXS_CCR_SMSB,
    CASE WHEN SegSMS LIKE 'SMSC' THEN 1 ELSE 0 END AS iXS_CCR_SMSC,
    CASE WHEN SegSMS LIKE 'SMSD' THEN 1 ELSE 0 END AS iXS_CCR_SMSD
INTO #TabelaFinalCCR
FROM #TabelaFinal a;


/* =========================================================
   4. PMS - ELEGÍVEIS
========================================================= */
WITH BasePMS AS (
    SELECT
        cp.NIF,
        d.NumDossier,
        d.MntDivida,
        d.TxTAN,
        d.NumVencMensalidade,
        d.CodProdAlfa AS CodProduto,
        d.Plafond,
        d.MntMensalidade,
        r.DescricaoReferencia
    FROM armada.dbo.DM_Dossier d
    INNER JOIN armada.dbo.DM_ClienteProcesso cp ON cp.IdProcesso = d.IdProcesso
    LEFT JOIN armada.dbo.IDH_DM_PropostaIntegral pi ON pi.NumDossier = d.NumDossier
    LEFT JOIN armada.dbo.PAC_Dim_Referencias r ON r.IdReferencias = pi.IdTipoParceiro
    WHERE d.MntDivida > 0 
      AND d.TxTAN > 0 
      AND cp.NIF <> 0 
      AND d.NumVencMensalidade > 4
),
ContagensPMS AS (
    SELECT
        NIF,
        COUNT(DISTINCT NumDossier) AS QtdDossiersUniverso,
        COUNT(DISTINCT CASE WHEN CodProduto LIKE 'CA' AND NumVencMensalidade > 24 
                       THEN NumDossier END) AS QtdAutoElegivel,
        COUNT(DISTINCT CASE WHEN CodProduto <> 'CA' AND DescricaoReferencia <> 'Directo' 
                       AND NumVencMensalidade > 12 THEN NumDossier END) AS QtdParceriasElegivel
    FROM BasePMS
    GROUP BY NIF
),
Universo2PMS AS (
    SELECT
        cp.NIF,
        COUNT(DISTINCT d.NumDossier) AS QtdDossiersUniverso2
    FROM armada.dbo.DM_Dossier d
    INNER JOIN armada.dbo.DM_ClienteProcesso cp ON cp.IdProcesso = d.IdProcesso
    WHERE d.MntDivida > 0 AND cp.NIF <> 0
    GROUP BY cp.NIF
)
SELECT
    b.NIF,
    b.NumDossier,
    b.Plafond AS plafondmaisrecente,
    b.MntMensalidade,
    b.CodProduto
INTO #PMS
FROM BasePMS b
INNER JOIN ContagensPMS c ON c.NIF = b.NIF
INNER JOIN armada.dbo.DM_CampanhasElegibilidade_NIF n ON n.NIF = b.NIF
INNER JOIN Universo2PMS u2 ON u2.NIF = b.NIF
WHERE c.QtdDossiersUniverso = 1
  AND u2.QtdDossiersUniverso2 = 1
  AND (c.QtdAutoElegivel = 1 OR c.QtdParceriasElegivel = 1)
  AND n.iPMSPA_PM = 1
  AND b.CodProduto NOT IN ('RUC','CC')
  AND OrdemMinima = 1;


/* =========================================================
   5. UNIVERSO BASE COMPLETO
========================================================= */
-- 5.1. CRUZAMENTO INICIAL
SELECT DISTINCT 
    eleg.*,
    CASE WHEN s.NIF IS NOT NULL THEN MensMaxSolvabilidadeNEW 
         ELSE eleg.MensMaxSolvabilidade END AS MensMaxSolvabilidadeNEW,
    CASE WHEN s.NIF IS NOT NULL THEN excl_solvNEW 
         ELSE excl_solv END AS excl_solvNEW,
    CASE WHEN s.NIF IS NOT NULL THEN excl_DONew 
         ELSE excl_DO END AS excl_DONew,
        CASE WHEN s.NIF IS NOT NULL THEN excl_rendNEW 
         ELSE excl_rend END AS excl_rendNEW
INTO #Cruzamento
FROM armada.dbo.DM_CampanhasElegibilidade eleg
INNER JOIN #tab_max_dt_eleg g ON g.max_dt = eleg.IdDia
INNER JOIN armada.dbo.DM_CampanhasElegibilidade_NIF nif ON nif.NIF = eleg.NIF
INNER JOIN armada.dbo.DM_Dossier d ON d.NumDossier = eleg.NumDossierOrigem
LEFT JOIN #SemSolvabilidade s ON s.NIF = eleg.NIF
LEFT JOIN #PMS p ON p.NIF = eleg.NIF
WHERE NOT (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120);

-- 5.2. UNIVERSO COM FLAGS
SELECT DISTINCT
    eleg.*,
    LEFT(DtAbertura, 4) AS AnoGeracao,
    -- iPCD_PA_PM
    CASE WHEN eleg.excl_score = 1 OR eleg.excl_Financiavel = 1 
              OR eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1
              OR eleg.excl_analise = 1 OR eleg.excl_ultRecusa = 1 OR eleg.excl_UltAbertura = 1
              OR eleg.excl_reclamacao = 1 OR eleg.excl_RequerInvestigacao = 1 
              OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL)
              OR eleg.excl_prof_instavel = 1 or excl_TipoResidencia = 1 
              OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)
         THEN 0 ELSE 1 END AS iPCD_PA_PM,
    -- iPCD_PA_GM
    CASE WHEN eleg.excl_score_MI = 1 OR eleg.excl_Financiavel = 1 
              OR eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1
              OR eleg.excl_analise = 1 OR eleg.excl_ultRecusa = 1 OR eleg.excl_UltAbertura = 1
              OR eleg.excl_DSTI = 1 
              OR eleg.excl_reclamacao = 1 OR eleg.excl_RequerInvestigacao = 1 
              OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL) 
              OR eleg.excl_prof_instavel = 1 or excl_TipoResidencia = 1 
              OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)
         THEN 0 ELSE 1 END AS iPCD_PA_GM,
    -- iRUC_PA
    CASE WHEN eleg.excl_revolving = 1 OR eleg.excl_score = 1 OR eleg.excl_Financiavel = 1 
              OR  eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1
              OR eleg.excl_idade = 1 OR eleg.excl_analise = 1 OR eleg.excl_ultRecusa = 1
              OR eleg.excl_UltAbertura = 1 or excl_TipoResidencia = 1 
              OR eleg.excl_reclamacao = 1 OR eleg.excl_RequerInvestigacao = 1
              OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL)
              OR eleg.excl_prof_instavel = 1 
              OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)
         THEN 0 ELSE 1 END AS iRUC_PA,
    iCCR, iAP, iAD, iAI, ScoreApetencia,
    -- iPMS_PA_PM
    CASE WHEN (eleg.excl_score = 1 OR eleg.excl_Financiavel = 1 
               OR eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1
               OR eleg.excl_analise = 1 OR eleg.excl_ultRecusa = 1 OR eleg.excl_UltAbertura = 1
               OR eleg.excl_reclamacao = 1 OR eleg.excl_RequerInvestigacao = 1 
               OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL) 
               OR eleg.excl_prof_instavel = 1 or excl_TipoResidencia = 1 
               OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)) or AtividadeCredito <> 'Ativo'
               OR p.NIF IS NULL or eleg.Ordem <> 1
         THEN 0 ELSE 1 END AS iPMS_PA_PM,
     -- iPMS_PA_MI
             CASE WHEN (eleg.excl_score = 1 OR eleg.excl_Financiavel = 1 
               OR eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1
               OR eleg.excl_analise = 1 OR eleg.excl_ultRecusa = 1 OR eleg.excl_UltAbertura = 1
               OR eleg.excl_reclamacao = 1 OR eleg.excl_RequerInvestigacao = 1 
               OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL) 
               OR eleg.excl_prof_instavel = 1 or excl_DSTI = 1 or excl_TipoResidencia = 1 
               OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)) or AtividadeCredito <> 'Ativo'
               OR p.NIF IS NULL or eleg.Ordem <> 1
         THEN 0 ELSE 1 END AS iPMS_PA_MI,
    -- iPCD_NPA
    CASE WHEN eleg.excl_Financiavel = 1 OR eleg.excl_removidos = 1 
              OR eleg.excl_solvNEW = 1 or eleg.excl_idade = 1 OR eleg.MesesUltAbertura < 4
              OR eleg.excl_analise = 1 OR eleg.MesesUltRecusa < 4 
              OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL) OR eleg.excl_Nacionalidade = 1
              OR eleg.excl_reclamacao = 1 OR eleg.excl_tiporesidencia = 1 or excl_TipoResidencia = 1 
              OR eleg.excl_prof_instavel = 1 OR AtividadeCredito = 'Terminado'
              OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)
         THEN 0 ELSE 1 END AS iPCD_NPA,
    -- iRUC_NPA
    CASE WHEN eleg.excl_Financiavel = 1 OR eleg.excl_ENI = 1 OR eleg.excl_removidos = 1 or excl_TipoResidencia = 1 
              OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1 OR eleg.excl_analise = 1
              OR eleg.MesesUltAbertura < 4 OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL)
              OR eleg.excl_DONew = 1 OR eleg.excl_Nacionalidade = 1 OR eleg.excl_reclamacao = 1
              OR eleg.excl_tiporesidencia = 1 OR eleg.excl_prof_instavel = 1 OR eleg.MesesUltRecusa < 4
              OR (eleg.excl_revolving = 1) OR AtividadeCredito = 'Terminado'
              OR (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120)
         THEN 0 ELSE 1 END AS iRUC_NPA,
    -- iTerminadosRUC
    CASE WHEN eleg.excl_Financiavel = 1 OR eleg.excl_ENI = 1 OR eleg.excl_removidos = 1 or excl_TipoResidencia = 1 
              OR eleg.excl_solvNEW = 1 or (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL)
              OR eleg.excl_idade = 1 OR eleg.excl_analise = 1
              OR eleg.excl_revolving = 1 OR eleg.excl_Nacionalidade = 1
              OR eleg.excl_reclamacao = 1 OR eleg.excl_tiporesidencia = 1 OR eleg.MesesUltRecusa < 4
              OR eleg.excl_prof_instavel = 1 OR AtividadeCredito <> 'Terminado'
         THEN 0 ELSE 1 END AS iTerminadosRUC,
    -- iTerminadosPCD
    CASE WHEN eleg.excl_Financiavel = 1 
              OR eleg.excl_removidos = 1 OR eleg.excl_solvNEW = 1 OR eleg.excl_idade = 1 or excl_TipoResidencia = 1 
              OR eleg.excl_analise = 1 OR excl_nacionalidade = 1 
              OR (excl_IncidentesBancarios = 1 and (max_dtCRC >= CONVERT(INT, FORMAT(DATEADD(MONTH, 3, CONVERT(DATE, CAST(eleg.DtUltCRC AS CHAR(8)))), 'yyyyMMdd'))) and DtUltCRC IS NOT NULL) OR eleg.excl_Nacionalidade = 1
              OR eleg.excl_reclamacao = 1 OR eleg.excl_tiporesidencia = 1 OR eleg.MesesUltRecusa < 4
              OR eleg.excl_prof_instavel = 1 OR AtividadeCredito <> 'Terminado'
         THEN 0 ELSE 1 END AS iTerminadosPCD
INTO #Universo
FROM #Cruzamento eleg
CROSS JOIN #tab_max_dt_CRC c
INNER JOIN #tab_max_dt_eleg g ON g.max_dt = eleg.IdDia
INNER JOIN armada.dbo.DM_CampanhasElegibilidade_NIF nif ON nif.NIF = eleg.NIF
INNER JOIN armada.dbo.DM_Dossier d ON d.NumDossier = eleg.NumDossierOrigem
LEFT JOIN #SemSolvabilidade s ON s.NIF = eleg.NIF
LEFT JOIN #PMS p ON p.NIF = eleg.NIF
WHERE NOT (AtividadeCredito LIKE 'Inativo' AND MesesInatividadeMinimo > 120);

/* =========================================================
   6. INFORMAÇÕES DE CAMPANHA E PRESSÃO COMERCIAL
========================================================= */
WITH CTE_UltimaCampanha AS (
    SELECT
        ec.NumContribuinte AS NIF,
        cd.Descricao,
                        CASE WHEN cd.Descricao LIKE '%R+1%' THEN 'iXS_RUC_R+1'
              WHEN cd.Descricao LIKE '%TOP%' AND cd.IdProduto = 25 THEN 'iXS_PCDPMTOP'
        WHEN cd.Descricao LIKE '%INA%' AND cd.IdProduto = 25 THEN 'iXS_PCD_Inativos_1T'
        WHEN cd.Descricao LIKE '%LAR%' AND cd.IdProduto = 25 THEN 'iXS_PCDPMHabit'
        WHEN cd.Descricao LIKE '%PARCERIAS%' AND cd.IdProduto = 25 THEN 'iXS_PCDPM_Parcerias'
        WHEN (cd.Descricao LIKE '%Auto%' or cd.Descricao LIKE '%MOTO%') and cd.IdProduto = 25 THEN 'iXS_PCD_Auto'
        WHEN cd.Descricao LIKE '%PLAFONDM%' AND cd.IdProduto = 23 THEN 'iXS_RUC_PlafondMinimo'

        WHEN cd.Descricao Like 'CSPCDPA_APT_DIG_JAN23' or cd.Descricao Like 'CSPCDPA_APT_DIG_DEZ22' or cd.Descricao LIKE 'CSPCDPA_APT_DIG' or (cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%1TIT%' or cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%' or cd.Descricao LIKE 'CSPCDPA_BYSIDE') AND cd.IdProduto = 25 AND cd.Tipo = 4615) THEN 'iXS_PCDPM_1T'
        WHEN cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%2TIT%' or cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%') AND cd.IdProduto = 25 AND cd.Tipo = 4615 THEN 'iXS_PCDPM_2T'

        WHEN (cd.Descricao LIKE 'CSRUCAUTO_CV') or cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%1TIT%' or cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.IdProduto = 23 AND cd.Tipo = 4615 THEN 'iXS_RUC_1T'
        WHEN cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%2TIT%' or cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%') AND cd.IdProduto = 23 AND cd.Tipo = 4615 THEN 'iXS_RUC_2T'

        WHEN (cd.Descricao LIKE '%CPAY%' AND cd.Descricao LIKE '%NPA%' AND cd.IdProduto = 25) or cd.Descricao LIKE '%COMBO_CPAY%' or cd.Descricao LIKE 'CSPCD_PAY' or cd.Descricao LIKE 'CSPCD_CPAY'  or cd.Descricao LIKE '%CSPCDCPAY_VOUCHER%'  THEN 'iXS_PCD_NPA_CPay'
        WHEN (cd.Descricao LIKE '%CPAY%' AND cd.Descricao LIKE '%NPA%' AND cd.IdProduto = 23) or cd.Descricao LIKE '%RUCCPAY%' or cd.Descricao LIKE 'CSRUC_CPAY' THEN 'iXS_RUC_NPA_CPay'

        WHEN (cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.Descricao LIKE '%NPA%' AND cd.IdProduto = 25) or cd.Descricao LIKE 'CSPCDSITE_NPA' or cd.Descricao LIKE 'CSPCD_NPA' THEN 'iXS_PCD_NPA_1T'
        WHEN cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%') AND cd.Descricao LIKE '%NPA%' AND cd.IdProduto = 25 THEN 'iXS_PCD_NPA_2T'
        WHEN (cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.Descricao LIKE '%NPA%' AND cd.IdProduto = 23) or cd.Descricao LIKE 'CSRUCSITE_NPA' or cd.Descricao LIKE 'CSRUC_NPA' THEN 'iXS_RUC_NPA_1T'
        WHEN cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%')  AND cd.IdProduto = 23 THEN 'iXS_RUC_NPA_2T'

        WHEN cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.Descricao LIKE '%PA%' AND cd.IdProduto = 25 THEN 'iXS_Terminados_PA_1T'
        WHEN cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%')  AND cd.Descricao LIKE '%PA%' AND cd.IdProduto = 25 THEN 'iXS_Terminados_PA_2T'
        WHEN cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.Descricao LIKE '%CAPT%' AND cd.IdProduto = 25 THEN 'iXS_Terminados_Captacao_1T'
        WHEN cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%')  AND cd.Descricao LIKE '%CAPT%' AND cd.IdProduto = 25 THEN 'iXS_Terminados_Captacao_2T'
        WHEN (cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%1T%' or cd.Descricao LIKE '%T1%') AND cd.IdProduto = 25) or cd.Descricao LIKE 'CSTERM' THEN 'iXS_Terminados_NPA_1T'
        WHEN cd.Descricao LIKE '%TERM%' AND (cd.Descricao LIKE '%2T%' or cd.Descricao LIKE '%T2%')  AND cd.IdProduto = 25 THEN 'iXS_Terminados_NPA_2T'

        WHEN cd.IdProduto = 28 and cd.Descricao LIKE '%MI%' THEN 'iXS_PMS_MI'
        WHEN cd.IdProduto = 28 and cd.Descricao LIKE '%PM%' THEN 'iXS_PMS_PM'

        WHEN (cd.Descricao LIKE '%SEG_A%' AND cd.IdProduto = 22) or cd.Descricao LIKE 'CSCCR' or cd.Descricao LIKE '%CCR_SMSA%' or cd.Descricao LIKE 'CSCCR_A' or cd.Descricao LIKE '%CCR_SMSA_C2C%' or cd.Descricao LIKE 'CCR_Geral_AtivCofidis' or cd.Descricao LIKE 'CSCCR_LOJ' THEN 'iXS_CCR_SMSA'
        WHEN (cd.Descricao LIKE '%SEG_B%' or cd.Descricao LIKE '%CCR_SMSB%' or cd.Descricao LIKE 'CSCCR_B') AND cd.IdProduto = 22 THEN 'iXS_CCR_SMSB'
        WHEN (cd.Descricao LIKE '%SEG_C%' or cd.Descricao LIKE '%CCR_SMSC%' or cd.Descricao LIKE 'CSCCR_C') AND cd.IdProduto = 22 THEN 'iXS_CCR_SMSC'
        WHEN (cd.Descricao LIKE '%SEG_D%' or cd.Descricao LIKE '%CCR_SMSD%' or cd.Descricao LIKE 'CSCCR_D') AND cd.IdProduto = 22 THEN 'iXS_CCR_SMSD'
        WHEN cd.Descricao LIKE '%T3%' AND cd.IdProduto = 25 THEN 'iXS_RUC_NPA_3T'
        WHEN (cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%SITE%') AND cd.IdProduto = 25 AND cd.Tipo = 4615) or (cd.Descricao LIKE '%PCD%' AND (cd.Descricao LIKE '%SUD%') AND cd.IdProduto = 25 AND cd.Tipo = 4615) THEN 'iXS_PCDPM_1T'
        WHEN (cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%SITE%') AND cd.IdProduto = 23 AND cd.Tipo = 4615) or cd.Descricao LIKE '%CSRUC_SITE%' or (cd.Descricao LIKE '%RUC%' AND (cd.Descricao LIKE '%SUD%') AND cd.IdProduto = 23 AND cd.Tipo = 4615) or cd.Descricao LIKE '%CSRUCPA%' THEN 'iXS_RUC_1T'
        ELSE NULL
        END as iTipoCampanha,
        p.CodProduto AS UltimoProdutoProposto,
        ec.DtCarregamento,
        ec.MontanteProposto,
        CASE 
            WHEN cd.Tipo = 4615 THEN 'PA'
            WHEN cd.Tipo = 528 THEN 'NPA'
            ELSE 'NUNCA RECEBEU' 
        END AS iCampanha,
        ROW_NUMBER() OVER (PARTITION BY ec.NumContribuinte ORDER BY ec.DtCarregamento DESC) AS RN
    FROM armada.dbo.DM_EnviosCampanhas ec
    INNER JOIN armada.dbo.DM_CampanhasDefinicao cd ON cd.IdCampanha = ec.IdCampanha
    LEFT JOIN armada.dbo.IDH_Dim_Produto p ON p.IdProduto = cd.IdProduto
    WHERE cd.Tipo IN (4615, 528) and cd.Descricao NOT LIKE '%ELEG%' and cd.Descricao NOT LIKE '%PRM%'
),
CTE_Pressao AS (
    SELECT
        ec.NumContribuinte AS NIF,
        COUNT(DISTINCT ec.IdCampanha) AS PressaoComercial
    FROM armada.dbo.DM_EnviosCampanhas ec
    INNER JOIN armada.dbo.DM_CampanhasDefinicao cd ON cd.IdCampanha = ec.IdCampanha
    WHERE ec.DtCarregamento >= DATEADD(YEAR, -1, GETDATE())
      AND cd.Tipo IN (4615, 528)
    GROUP BY ec.NumContribuinte
),
CTE_RUCPlafond AS (
    SELECT
        ec.NumContribuinte AS NIF,
        COUNT(DISTINCT ec.IdCampanha) AS QtdRUCPlafond
    FROM armada.dbo.DM_EnviosCampanhas ec
    INNER JOIN armada.dbo.DM_CampanhasDefinicao cd ON cd.IdCampanha = ec.IdCampanha
    WHERE ec.DtCarregamento >= DATEADD(MONTH, -3, GETDATE())
      AND cd.Descricao LIKE 'CSRUC_PLAFONDM_CPAY'
    GROUP BY ec.NumContribuinte
)
SELECT
    u.*,
    COALESCE(c.Descricao, 'Sem Campanha') AS DescricaoUltimaCampanha,
    COALESCE(c.iTipoCampanha, 'Sem Campanha') AS TipoUltimaCampanha,
    COALESCE(c.iCampanha, 'Sem Campanha') AS TipologiaUltimaCampanha,
    c.MontanteProposto AS MontanteUltimaCampanha,
    c.UltimoProdutoProposto AS ProdutoUltimaCampanha,
    FORMAT(c.DtCarregamento, 'yyyyMM') AS IdMesUltimaCampanha,
    CONVERT(VARCHAR(8), c.DtCarregamento, 112) AS DtUltimaCampanha,
    pc.PressaoComercial,
    CASE WHEN pp.QtdRUCPlafond < 2 THEN 1 ELSE 0 END AS iPressaoPlafondMinimo
INTO #BaseConsolidada
FROM #Universo u
LEFT JOIN CTE_UltimaCampanha c ON c.NIF = u.NIF AND c.RN = 1
LEFT JOIN CTE_Pressao pc ON pc.NIF = u.NIF
LEFT JOIN CTE_RUCPlafond pp ON pp.NIF = u.NIF;

/* =========================================================
   7. FLAGS ADICIONAIS (LAR, AUTO, MOTO, PLAFOND)
========================================================= */
SELECT
    b.*,
    pm.MensalidadeCofidisPay,
    CASE WHEN hab.NIF IS NOT NULL OR lar.NIF IS NOT NULL THEN 1 ELSE 0 END AS iLar,
    CASE WHEN cli.IdCliente IS NOT NULL THEN 1 ELSE 0 END AS iAuto,
    CASE WHEN moto.NIF IS NOT NULL THEN 1 ELSE 0 END AS iMoto,
    CASE WHEN pm.NIF IS NOT NULL
          AND SubcanalNegocioAtual LIKE 'CofidisPay'
          AND (AtividadeCredito LIKE 'Ativo' OR (AtividadeCredito = 'Terminado' AND MesesTerminoMinimo < 2))
          AND (iPressaoPlafondMinimo <= 1 or iPressaoPlafondMinimo IS NULL)
         THEN 1 ELSE 0 END AS iPlafondMinimo,
    CASE WHEN DataFimMotivo >= CONVERT(VARCHAR(8), DATEADD(DAY, -7, CAST(GETDATE() AS DATE)), 112) THEN 1 ELSE 0 end as iQuarentenaFinanciabilidade  
INTO #BaseFinal
FROM #BaseConsolidada b
LEFT JOIN (
    SELECT CAST(idEnt AS BIGINT) AS NIF
    FROM armada.dbo.CRC_DM_Disseminacao5G
    WHERE dissTpInst = '0110' AND CAST(dissMontTotal AS FLOAT) > 0
    GROUP BY idEnt
) hab ON hab.NIF = b.NIF
LEFT JOIN (
    SELECT DISTINCT cp.NIF
    FROM armada.dbo.IDH_DM_PropostaIntegral pi
    INNER JOIN armada.dbo.DM_ClienteProcesso cp ON pi.IdProcesso = cp.IdProcesso
    INNER JOIN armada.dbo.PAC_Dim_Referencias pref ON pi.IdBemAFinanciar = pref.IdReferencias
    WHERE pi.Canal = 'Parcerias'
      AND pref.IdReferencias IN (
        104795,104648,104447,104793,104796,104797,104802,104804,
        104641,104456,104446,104448,104803,104808,104806,104646,
        104647,104791,104629,104786,104792,104798
      )
) lar ON lar.NIF = b.NIF
LEFT JOIN armada.dbo.IDH_DM_Cliente cli ON cli.IdCliente = b.IdCliente
       AND cli.IdDataNascimento <= CAST(FORMAT(DATEADD(YEAR, -25, GETDATE()), 'yyyyMMdd') AS INT)
LEFT JOIN (
    SELECT DISTINCT cp.NIF
    FROM armada.dbo.IDH_DM_PropostaIntegral pi
    INNER JOIN armada.dbo.DM_ClienteProcesso cp ON cp.IdProcesso = pi.IdProcesso
    WHERE pi.IdFinalidadeComercial = 19
      AND pi.DtPedido < CAST(FORMAT(DATEADD(YEAR, -3, GETDATE()), 'yyyyMMdd') AS INT)
) moto ON moto.NIF = b.NIF
LEFT JOIN (
    SELECT
        cp.NIF,
        MAX(MntMensalidadeCnt) AS MensalidadeCofidisPay
    FROM armada.dbo.DM_Dossier d
    INNER JOIN armada.dbo.DM_ClienteProcesso cp ON cp.IdProcesso = d.IdProcesso
    WHERE CodProdAlfa LIKE 'SPF'
      AND (LEFT(DtFimContrato, 6) = FORMAT(GETDATE(), 'yyyyMM')
           OR (LEFT(DtUltPagamento, 6) > LEFT(CONVERT(VARCHAR(8), DATEADD(MONTH, -2, CAST(GETDATE() AS DATE)), 112), 6) 
               AND MntDivida <= 0))
    GROUP BY cp.NIF
) pm ON pm.NIF = b.NIF
LEFT JOIN (SELECT IdTitular as IdCliente, MAX(DataFimMotivo) as DataFimMotivo
FROM armada.dbo.DM_CriteriosNFinanciavelTitular
WHERE DataFimMotivo <> 20301231 and CodMotivo IN ('1','3','4','6','8','10','12','20','26','27A','27B','27C','27D','27E','27F','27G','27H','28','10A','13')
GROUP BY IdTitular) as nf 
on nf.IdCliente = b.IdCliente;


/* =========================================================
   8. PROPOSTAS COMERCIAIS
========================================================= */
-- 8.1. RUC
WITH CTE_RUC AS (
    SELECT a.*, d.Descricao,
           ROW_NUMBER() OVER (PARTITION BY AMOUNT ORDER BY OfferTarificationId DESC, a.DeadlineWithInsurance DESC) AS rn
    FROM armada.dbo.ST_DM_CondicoesSimulacao a
    LEFT JOIN armada.dbo.DM_Oferta b ON a.idoffer = b.idOferta
    LEFT JOIN armada.dbo.IDH_Dim_Produto c ON b.Produto = c.CodProdutoDAH
    LEFT JOIN armada.dbo.IDH_Dim_Finalidade d ON a.PURPOSEID = d.IDFinalidade
    WHERE c.CodProduto = 'RUC' AND a.AMOUNT IN (1000,1500,2000,3000,4000)
)
SELECT * INTO #proposta_comercial_RUC FROM CTE_RUC WHERE rn = 1;

-- 8.2. PCD Intermediários
WITH CTE_PCD_Intermedios AS (
    SELECT a.*, d.Descricao,
           ROW_NUMBER() OVER (PARTITION BY AMOUNT ORDER BY OfferTarificationId DESC, a.DeadlineWithInsurance DESC) AS rn
    FROM armada.dbo.ST_DM_CondicoesSimulacao a
    LEFT JOIN armada.dbo.DM_Oferta b ON a.idoffer = b.idOferta
    LEFT JOIN armada.dbo.IDH_Dim_Produto c ON b.Produto = c.CodProdutoDAH
    LEFT JOIN armada.dbo.IDH_Dim_Finalidade d ON a.PURPOSEID = d.IDFinalidade
    WHERE a.AMOUNT IN (9000,10000,11000,12000,13000,14000,15000)
      AND DEADLINEWITHINSURANCE = 84 AND IDFinalidade = 6
)
SELECT * INTO #proposta_comercial_PCD_Intermedios FROM CTE_PCD_Intermedios WHERE rn = 1;

-- 8.3. PCD
WITH CTE_PCD AS (
    SELECT a.*, d.Descricao,
           ROW_NUMBER() OVER (PARTITION BY AMOUNT ORDER BY OfferTarificationId DESC, a.DeadlineWithInsurance DESC) AS rn
    FROM armada.dbo.ST_DM_CondicoesSimulacao a
    LEFT JOIN armada.dbo.DM_Oferta b ON a.idoffer = b.idOferta
    LEFT JOIN armada.dbo.IDH_Dim_Produto c ON b.Produto = c.CodProdutoDAH
    LEFT JOIN armada.dbo.IDH_Dim_Finalidade d ON a.PURPOSEID = d.IDFinalidade
    WHERE a.AMOUNT IN (2500,3000,3500,4000,4500,5000,6000,6500,7000,7500,8000)
      AND DEADLINEWITHINSURANCE = 84 AND IDFinalidade = 6
)
SELECT * INTO #proposta_comercial_PCD FROM CTE_PCD WHERE rn = 1;

-- 8.4. PCD Auto
WITH CTE_PCDAuto AS (
    SELECT a.*, d.Descricao,
           ROW_NUMBER() OVER (PARTITION BY AMOUNT ORDER BY OfferTarificationId DESC, a.DeadlineWithInsurance DESC) AS rn
    FROM armada.dbo.ST_DM_CondicoesSimulacao a
    LEFT JOIN armada.dbo.DM_Oferta b ON a.idoffer = b.idOferta
    LEFT JOIN armada.dbo.IDH_Dim_Produto c ON b.Produto = c.CodProdutoDAH
    LEFT JOIN armada.dbo.IDH_Dim_Finalidade d ON a.PURPOSEID = d.IDFinalidade
    WHERE ((AMOUNT IN (8000,9000) AND DEADLINEWITHINSURANCE = 96)
        OR (AMOUNT IN (10000,11000,12000,13000,14000,15000,16000) AND DEADLINEWITHINSURANCE = 120)
        OR (AMOUNT IN (5000,6000,7000) AND DEADLINEWITHINSURANCE = 84))
      AND IDFinalidade = 2
)
SELECT * INTO #proposta_comercial_PCDAuto FROM CTE_PCDAuto WHERE rn = 1;

SELECT DISTINCT
CAST(a.AMOUNT AS INT) AS MONTANTE,
CAST(a.TAN AS DECIMAL(10,2)) AS TAN,
CAST(a.TAEG AS DECIMAL(10,1)) AS TAEG,
CAST(a.DeadlineWithInsurance AS INT) AS Prazo,
CAST(a.MONTHLYPAYWITHOUTINSURANCE AS DECIMAL(10,2)) AS MM,
CAST(a.Insurance AS DECIMAL(10,2)) AS Seguro,
CAST(a.MTIC AS DECIMAL(10,2)) AS MTIC
INTO #proposta_comercial_PMS
FROM DataExperimental_DICV.dbo.TabelaFinanceira_PMS a;


-- 8.6. Cálculo PMS
WITH base_calculo_pms AS (
    SELECT
        a.NIF,
        a.RendSolv,
        p.NumDossier,
        a.DespesasSolv,
        a.PrestMes,
        a.PrestMesCofidis,
        a.MntDivida AS encours_cliente,
        p.plafondmaisrecente,
        b.MONTANTE,
        b.PRAZO,
        b.TAN,
        b.TAEG,
        b.MM,
        b.Seguro,
        b.MTIC,
        (MensMaxSolvabilidade + a.PrestMesCofidis) AS MM_Max,
        (b.MM - a.PrestMesCofidis) AS aumento_mensalidade,
        (b.MONTANTE - a.MntDivida) AS liquidez_adicional
    FROM armada.dbo.DM_CampanhasElegibilidade a
    INNER JOIN #tab_max_dt_eleg e ON e.max_dt = a.IdDia
    INNER JOIN #PMS p ON p.NIF = a.NIF
    CROSS JOIN #proposta_comercial_PMS b
),
regras_alvo_pms AS (
    SELECT *,
        CASE
            WHEN encours_cliente <= 4000 THEN encours_cliente + 2000
            WHEN encours_cliente > 4000 AND encours_cliente <= 8000 THEN encours_cliente * 2
            WHEN encours_cliente > 8000 THEN encours_cliente - MONTANTE
        END AS montante_alvo
    FROM base_calculo_pms
),
metricas_pms AS (
    SELECT *,
        ABS(MONTANTE - montante_alvo) AS dif_montante,
        (liquidez_adicional - 1000) AS dif_liquidez_min,
        (liquidez_adicional - (0.20 * encours_cliente)) AS dif_liquidez_pct,
        (MONTANTE - (2 * plafondmaisrecente)) AS dif_plafond,
        (MM_Max - MM) AS dif_MM
    FROM regras_alvo_pms
),
filtrada_pms AS (
    SELECT *
    FROM metricas_pms
    WHERE dif_MM > 0
      AND aumento_mensalidade < 80
      AND dif_liquidez_min >= 0
      AND dif_liquidez_pct >= 0
)
SELECT
    NIF, t.NumDossier as param2, liquidez_adicional as param3, aumento_mensalidade as param4, plafondmaisrecente as param5, PrestMesCofidis as param6, d.MesesPrazo as param7, DATEDIFF(MONTH, GETDATE(), CAST(CAST(DtFimContrato as CHAR(8)) AS DATE)) as param8, TxTan as param9, d.TAEG as param10,
    MntDivida as param11,     MONTANTE, Montante as param12, MM as param13, PRAZO as param14, TAN as param15, t.TAEG as param16, Seguro as Param17, MTIC as param18, NIF as external_id, MM + Seguro as param21
INTO #ElegiveisPMS
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY NIF ORDER BY dif_montante ASC, MONTANTE DESC) AS rn_final
    FROM filtrada_pms
) t
LEFT JOIN armada.dbo.DM_Dossier d on d.NumDossier = t.NumDossier
WHERE rn_final = 1;


/* =========================================================
   8.1. MATCH TERMINADOS
========================================================= */
DROP TABLE IF EXISTS #ClientesTerminados;
WITH CTE_Dossier AS (
    SELECT a.numdossier, b.nif, Ordem, MntMensalidadeCnt, ROW_NUMBER() OVER (PARTITION BY b.NIF ORDER BY a.DtSitDossier DESC) AS RN
            from armada.dbo.dm_dossier a 
            left join armada.dbo.DM_ClienteProcesso b on a.numdossier = b.numdossier and b.TipoInterveniente='T'
            where CodSitDossier = 'T' or iDossierEmCurso=0)
SELECT DISTINCT 
    rf.NIF, 
    d.MntMensalidadeCnt as UltimaMensalidadeTerminados
INTO #ClientesTerminados
FROM #BaseFinal rf
LEFT JOIN CTE_Dossier d on d.NIF = rf.NIF AND d.RN = 1 
WHERE rf.AtividadeCredito LIKE 'Terminado'


/* =========================================================
   9. MATCHING FINANCEIRO
========================================================= */
SELECT
    bf.*,
    (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_RUC pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND bf.MensMaxSolvabilidadeNEW >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_RUC,
    (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_PCD pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND bf.MensMaxSolvabilidadeNEW >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_PCD,
    (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_PCD_Intermedios pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND bf.MensMaxSolvabilidadeNEW >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_PCD_MI,
    (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_PCDAuto pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND bf.MensMaxSolvabilidadeNEW >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_PCDAuto,
    (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_RUC pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND bf.MensalidadeCofidisPay >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_RUCPlafondMinimo,
         (SELECT TOP 1 pc.Amount
     FROM #proposta_comercial_PCD pc
     WHERE pc.MonthlyPayWithoutInsurance IS NOT NULL
       AND UltimaMensalidadeTerminados >= pc.MonthlyPayWithoutInsurance
     ORDER BY pc.Amount DESC) AS Melhor_Match_Terminados,
     c.Montante as Melhor_Match_CCD,
     p.MONTANTE as Melhor_Match_PMS
INTO #Resultado_Financeiro
FROM #BaseFinal bf
LEFT JOIN #TabelaFinalCCR c on c.NIF = bf.NIF 
LEFT JOIN #ElegiveisPMS p on p.NIF = bf.NIF
LEFT JOIN #ClientesTerminados as t on t.NIF = bf.NIF;

/* =========================================================
   1. Data máxima de elegibilidade
========================================================= */

---- BASE CLIENTES 
DROP TABLE IF EXISTS #ConcelhosAfectados;
SELECT DISTINCT NIF, Concelho
INTO #ConcelhosAfectados
FROM armada.dbo.DM_CampanhasElegibilidade_NIF
WHERE concelho IN (
'Abrantes', 'Alcanena', 'Alcobaça', 'Alvaiázere', 'Ansião', 'Batalha',
'Bombarral', 'Cadaval', 'Caldas da Rainha', 'Cantanhede',
'Castanheira de Pera', 'Castelo Branco', 'Coimbra', 'Condeixa-a-Nova',
'Constância', 'Covilhã', 'Entroncamento', 'Ferreira do Zêzere',
'Figueira da Foz', 'Figueiró dos Vinhos', 'Fundão', 'Góis', 'Golegã',
'Idanha-a-Nova', 'Leiria', 'Lourinhã', 'Lousã', 'Mação',
'Marinha Grande', 'Mealhada', 'Mira', 'Miranda do Corvo',
'Montemor-o-Velho', 'Nazaré', 'Óbidos', 'Oleiros', 'Ourém',
'Pampilhosa da Serra', 'Pedrogão Grande', 'Penacova', 'Penamacôr',
'Penela', 'Peniche', 'Pombal', 'Porto de Mós', 'Proença-a-Nova',
'Rio Maior', 'Santarém', 'Sardoal', 'Sertã', 'Soure', 'Tomar',
'Torres Novas', 'Torres Vedras', 'Vagos', 'Vila de Rei',
'Vila Nova da Barquinha', 'Vila Nova de Poiares',
'Vila Velha de Ródão', 'Águeda', 'Albergaria-a-Velha',
'Alcácer do Sal', 'Aveiro', 'Estarreja', 'Ílhavo', 'Murtosa',
'Ovar', 'Sever do Vouga'
);


/* =========================================================
   10. BASE DE ENVIO FINAL
========================================================= */
WITH BasePCDPMTOP AS (
    SELECT DISTINCT
        NIF,
        ROW_NUMBER() OVER (ORDER BY ScoreApetencia DESC) AS rn_PCDPMTOP
    FROM #Resultado_Financeiro
    WHERE iPCD_PA_PM = 1
      AND Melhor_Match_PCD IS NOT NULL
      AND Ordem = 1
      AND (iOptOut_Email = 0 OR iOptOut_SMS = 0)
      AND AtividadeCredito = 'Ativo'
      AND ScoreApetencia < 0
)
SELECT DISTINCT
    rf.NIF,
    rf.IdCliente,
    NIF.PrimeiroNome as PrimNome,
    CASE WHEN nif.Email LIKE '' THEN NULL
         WHEN iOptOut_Email = 0 THEN NIF.Email
         ELSE NULL END as Email,
    CASE WHEN nif.Telemovel LIKE '' THEN NULL
         WHEN nif.Telemovel = 0 THEN NULL
         WHEN LEN(nif.Telemovel) = 9 AND iOptOut_SMS = 0 THEN nif.Telemovel
         ELSE NULL END AS telemovel,
    CASE
        WHEN nif.Loja = 'SEDE' THEN 'LISBOA'
        WHEN nif.Loja = 'FARO' THEN 'SETUBAL'
        WHEN nif.Loja = 'CASTELO BRANCO' THEN 'COIMBRA'
        WHEN nif.Loja = 'EVORA' THEN 'SETUBAL'
        WHEN nif.Loja = 'MIRANDELA' THEN 'VISEU'
        WHEN nif.Loja = 'SINTRA' THEN 'LISBOA'
        WHEN nif.Loja = 'VIANA DO CASTELO' THEN 'BRAGA'
        WHEN nif.Loja = 'TERCEIRA' THEN 'AÇORES'
        WHEN nif.Loja like '' THEN 'LISBOA'
        ELSE nif.Loja
    END as NomeLoja,
    Ordem,
    IdMesUltimaCampanha,
    TipoUltimaCampanha,
    TipologiaUltimaCampanha,
    ProdutoUltimaCampanha,
    PressaoComercial,
    Melhor_Match_PCD,
    Melhor_Match_PCD_MI,
    Melhor_Match_RUC,
    Melhor_Match_PCDAuto,
    Melhor_Match_RUCPlafondMinimo,
    c.Montante as Melhor_Match_CCR,
    CASE WHEN AtividadeCredito LIKE 'Terminado' and Melhor_Match_Terminados is NULL THEN 8000
    WHEN AtividadeCredito LIKE 'Terminado' THEN Melhor_Match_Terminados
    ELSE NULL end as Melhor_Match_Terminados, 
    Melhor_Match_PMS, 
    iQuarentenaFinanciabilidade,
    -- PCD AUTO
    CASE WHEN iAuto = 1 AND (iPCD_PA_PM = 1 OR iPCD_PA_GM = 1)
              AND Melhor_Match_PCD IS NOT NULL AND Ordem = 1
              AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_PCD_Auto,
    -- PCDPM HABITACIONAL
    CASE WHEN iLar = 1 AND iPCD_PA_PM = 1
              AND Melhor_Match_PCD IS NOT NULL AND Ordem = 1
              AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_PCDPMHabit,
    -- PCDPM TOP
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND Ordem = 1 AND AtividadeCredito = 'Ativo'
              AND b.rn_PCDPMTOP <= 25000
         THEN 1 ELSE 0 END AS iXS_PCDPMTOP,
    -- PCDPM PARCERIAS
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND CanalNegocioAtual = 'Parcerias'
              AND Ordem = 1 AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_PCDPM_Parcerias,
    -- PCDPM 1T e 2T
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND Ordem = 1 AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_PCDPM_1T,
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND Ordem = 2 AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_PCDPM_2T,
    -- RUC 1T e 2T
    CASE WHEN iRUC_PA = 1 AND Melhor_Match_RUC IS NOT NULL
              AND Ordem = 1 AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_RUC_1T,
    CASE WHEN iPlafondMinimo = 1 and (iRUC_NPA = 1 or iTerminadosRUC = 1) THEN 1 ELSE 0 END AS iXS_RUC_PlafondMinimo,
    CASE WHEN iPlafondMinimo = 1 and (iRUC_NPA = 1 or iTerminadosRUC = 1) and rucp.MM = MensalidadeCofidisPay THEN 'IGUAL'
    WHEN iPlafondMinimo = 1 and (iRUC_NPA = 1 or iTerminadosRUC = 1) and rucp.MM < MensalidadeCofidisPay THEN 'INFERIOR'
    ELSE NULL END as iMensalidadeIgualRUCPlafondMinimo, 
    CASE WHEN iRUC_PA = 1 AND Melhor_Match_RUC IS NOT NULL
              AND Ordem = 2 AND AtividadeCredito = 'Ativo'
         THEN 1 ELSE 0 END AS iXS_RUC_2T,
    -- PCD INATIVOS 1T e 2T
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND Ordem = 1 AND AtividadeCredito = 'Inativo'
         THEN 1 ELSE 0 END AS iXS_PCD_Inativos_1T,
    CASE WHEN iPCD_PA_PM = 1 AND Melhor_Match_PCD IS NOT NULL
              AND Ordem = 2 AND AtividadeCredito = 'Inativo'
         THEN 1 ELSE 0 END AS iXS_PCD_Inativos_2T,
    -- RUC_NPA 1T e 2T
    CASE WHEN iRUC_NPA = 1 AND ((iPCD_PA_PM = 0 AND iPCD_PA_GM = 0) OR Melhor_Match_PCD IS NULL)
              AND (iRUC_PA = 0 OR Melhor_Match_RUC IS NULL)
              AND SubcanalNegocioAtual NOT LIKE 'SPF'
              AND AtividadeCredito <> 'Terminado'
              AND iPlafondMinimo = 0 AND Ordem = 1
         THEN 1 ELSE 0 END AS iXS_RUC_NPA_1T,
    CASE WHEN iRUC_NPA = 1 AND ((iPCD_PA_PM = 0 AND iPCD_PA_GM = 0) OR Melhor_Match_PCD IS NULL)
              AND (iRUC_PA = 0 OR Melhor_Match_RUC IS NULL)
              AND SubcanalNegocioAtual NOT LIKE 'SPF'
              AND AtividadeCredito <> 'Terminado'
              AND iPlafondMinimo = 0 AND Ordem = 2
         THEN 1 ELSE 0 END AS iXS_RUC_NPA_2T,
    -- PCD_NPA 1T e 2T
    CASE WHEN iPCD_NPA = 1 AND ((iPCD_PA_PM = 0 AND iPCD_PA_GM = 0) OR Melhor_Match_PCD IS NULL)
              AND (iRUC_PA = 0 OR Melhor_Match_RUC IS NULL)
              AND SubcanalNegocioAtual NOT LIKE 'SPF'
              AND AtividadeCredito <> 'Terminado'
              AND iPlafondMinimo = 0 AND Ordem = 1
         THEN 1 ELSE 0 END AS iXS_PCD_NPA_1T,
    CASE WHEN iPCD_NPA = 1 AND ((iPCD_PA_PM = 0 AND iPCD_PA_GM = 0) OR Melhor_Match_PCD IS NULL)
              AND (iRUC_PA = 0 OR Melhor_Match_RUC IS NULL)
              AND SubcanalNegocioAtual NOT LIKE 'SPF'
              AND AtividadeCredito <> 'Terminado'
              AND iPlafondMinimo = 0 AND Ordem = 2
         THEN 1 ELSE 0 END AS iXS_PCD_NPA_2T,
    -- TERMINADOS NPA 1T e 2T
    CASE WHEN (iTerminadosPCD = 1)
              AND (MesesTerminoMinimo < 24 and AtividadeCredito LIKE 'Terminado')
              AND Ordem = 1 AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_Terminados_NPA_1T,
    CASE WHEN (iTerminadosPCD = 1)
              AND (MesesTerminoMinimo < 24 and AtividadeCredito LIKE 'Terminado')
              AND Ordem = 2 AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_Terminados_NPA_2T,
    -- TERMINADOS CAPTAÇÃO 1T e 2T
    CASE WHEN (iTerminadosPCD = 1 OR iTerminadosRUC = 1)
              AND Ordem = 1
              AND (MesesTerminoMinimo >= 24 and AtividadeCredito LIKE 'Terminado')
              AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_Terminados_Captacao_1T,
    CASE WHEN (iTerminadosPCD = 1 OR iTerminadosRUC = 1)
              AND Ordem = 2
              AND (MesesTerminoMinimo >= 24 and AtividadeCredito LIKE 'Terminado')
              AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_Terminados_Captacao_2T,
    -- TERMINADOS PA 1T e 2T
    CASE WHEN ((iPCD_PA_PM = 1 OR iPCD_PA_GM = 1 OR iRUC_PA = 1) AND Melhor_Match_PCD IS NOT NULL)
              AND AtividadeCredito = 'Terminado'
              AND MesesTerminoMinimo BETWEEN 0 AND 2
              AND iPlafondMinimo = 0 AND Ordem = 1
         THEN 1 ELSE 0 END AS iXS_Terminados_PA_1T,
    CASE WHEN ((iPCD_PA_PM = 1 OR iPCD_PA_GM = 1 OR iRUC_PA = 1) AND Melhor_Match_PCD IS NOT NULL)
              AND AtividadeCredito = 'Terminado'
              AND MesesTerminoMinimo BETWEEN 0 AND 2
              AND iPlafondMinimo = 0 AND Ordem = 2
         THEN 1 ELSE 0 END AS iXS_Terminados_PA_2T,
    -- RUC/PCD NPA COFIDISPAY
    CASE WHEN iRUC_NPA = 1 AND((iPCD_PA_PM = 1 OR iPCD_PA_GM = 1 OR iRUC_PA = 1) AND Melhor_Match_PCD IS NOT NULL)
              AND  (iRUC_PA = 0 OR Melhor_Match_RUC IS NULL) AND SubcanalNegocioAtual LIKE 'CofidisPay'
              AND AtividadeCredito <> 'Terminado' AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_RUC_NPA_CPay,
    CASE WHEN iPCD_NPA = 1 AND ((iPCD_PA_PM = 0 AND iPCD_PA_GM = 0) AND Melhor_Match_PCD IS NOT NULL)
              AND SubcanalNegocioAtual LIKE 'CofidisPay'
              AND AtividadeCredito <> 'Terminado' AND iPlafondMinimo = 0
         THEN 1 ELSE 0 END AS iXS_PCD_NPA_CPay,
    -- PMS
    CASE WHEN p.NIF IS NOT NULL and p.param12 <= 8000 THEN 1 ELSE 0 END AS iXS_PMS_PM,
    CASE WHEN p.NIF IS NOT NULL and p.param12 > 8000 THEN 1 ELSE 0 END AS iXS_PMS_MI,
    COALESCE(c.iXS_CCR_SMSA, 0) as iXS_CCR_SMSA,
    COALESCE(c.iXS_CCR_SMSB, 0) as iXS_CCR_SMSB,
    COALESCE(c.iXS_CCR_SMSC, 0) as iXS_CCR_SMSC,
    COALESCE(c.iXS_CCR_SMSD, 0) as iXS_CCR_SMSD,
    COALESCE(c.iXS_CCR, 0) as iXS_CCR,
    iOptOut_Email,
    iOptOut_SMS,
    CASE WHEN app.IDCLIENTE IS NOT NULL THEN 1 ELSE 0 END as iApp
INTO #BaseEnvio
FROM #Resultado_Financeiro rf
LEFT JOIN BasePCDPMTOP b ON rf.NIF = b.NIF
LEFT JOIN #ElegiveisPMS p ON p.NIF = rf.NIF
LEFT JOIN #ConcelhosAfectados af on af.NIF = rf.NIF
LEFT JOIN (SELECT Montante, NIF, iXS_CCR_SMSA, iXS_CCR_SMSB, iXS_CCR_SMSC, iXS_CCR_SMSD, 1 as iXS_CCR 
           FROM #TabelaFinalCCR) c ON c.NIF = rf.NIF
INNER JOIN armada.dbo.DM_CampanhasElegibilidade_NIF nif ON nif.NIF = rf.NIF
LEFT JOIN (SELECT MonthlyPayWithoutInsurance as MM, Amount FROM #proposta_comercial_RUC pc) as rucp on rucp.Amount = rf.Melhor_Match_RUCPlafondMinimo
left join (
    select a.CLIENTID AS IDCLIENTE, 
           max(LOGINDATE) as UltLogin
    from armada.dbo.DM_CofidisPay_UsersLogins a 
    left join armada.dbo.DM_CofidisPay_CarteiraApp b on a.CLIENTID = b.IdCliente
    where HAVESUCCESS = 1 
      and ENTRYLOGAPPTYPEID = 3 
      and b.iTermosAceitesApp = 1 
      and left(convert(varchar(10),cast(DtTermosAceitesApp as date),112),8) <= LOGINDATE
      and LOGINDATE >= 20250609
    group by a.CLIENTID) as app on app.IdCliente = rf.IdCliente
WHERE af.NIF IS NULL;


/* =========================================================
   11. LIMPEZA FINAL
========================================================= */
DELETE FROM #BaseEnvio
WHERE (Email IS NULL AND telemovel IS NULL)  or iQuarentenaFinanciabilidade = 1 
   OR (iXS_PCD_Auto + iXS_PCDPMHabit + iXS_PCDPMTOP + iXS_PCDPM_Parcerias +
       iXS_PCDPM_1T + iXS_PCDPM_2T + iXS_RUC_1T + iXS_RUC_PlafondMinimo +
       iXS_RUC_2T + iXS_PCD_Inativos_1T + iXS_PCD_Inativos_2T + iXS_RUC_NPA_1T +
       iXS_RUC_NPA_2T + iXS_PCD_NPA_1T + iXS_PCD_NPA_2T + iXS_Terminados_NPA_1T +
       iXS_Terminados_NPA_2T + iXS_Terminados_Captacao_1T + iXS_Terminados_Captacao_2T +
       iXS_Terminados_PA_1T + iXS_Terminados_PA_2T + iXS_RUC_NPA_CPay +
       iXS_PCD_NPA_CPay + iXS_PMS_PM + iXS_PMS_MI + iXS_CCR) = 0;
"""

cursor.execute(query_base_envio)
conn.commit()

df_baseenvio = pd.read_sql("SELECT DISTINCT * FROM #BaseEnvio", conn)

print(f"✓ df_baseenvio carregado: {len(df_baseenvio):,} clientes")
print(f"  Colunas: {list(df_baseenvio.columns)}")


In [ ]:
# =============================================================================
# BLOCO 3: CARREGAMENTO DO PLANO DE CAMPANHAS (CSV)
# =============================================================================

def carregar_plano(path=PATH_PLANO):
    """Carrega df_plano a partir do CSV, com fallback para o ficheiro local."""
    _script_dir = os.getcwd()
    caminhos = [path, os.path.join(_script_dir, 'df_plano.csv'), 'df_plano.csv']
    for c in caminhos:
        if os.path.exists(c):
            df = pd.read_csv(c, sep=SEPARADOR, encoding=ENCODING)
            df.columns = [col.strip() for col in df.columns]
            df['data']       = pd.to_datetime(df['data'], dayfirst=True, errors='coerce')
            df['volumetria'] = pd.to_numeric(df['volumetria'], errors='coerce').fillna(0).astype(int)
            if 'tipo oferta' not in df.columns:
                df['tipo oferta'] = 'PCD'
            return df
    raise FileNotFoundError(
        f"Ficheiro de plano não encontrado em {path} nem em df_plano.csv"
    )

df_plano = carregar_plano()
print(f"✓ Plano carregado: {len(df_plano)} campanhas")
df_plano.head()


In [ ]:
# =============================================================================
# BLOCO 4: FUNÇÕES AUXILIARES
# =============================================================================

def calcular_tier_antiguidade(idmes_ultima_campanha, mes_atual):
    """
    Classifica um cliente pelo tempo sem receber campanha.

    Tier 1 (NULL / nunca contactado) → MÁXIMA PRIORIDADE
    Tier 2 (≥ 6 meses)               → ALTA PRIORIDADE
    Tier 3 (3 – 5 meses)             → MÉDIA PRIORIDADE
    Tier 4 (< 3 meses)               → BAIXA PRIORIDADE
    """
    if (
        pd.isna(idmes_ultima_campanha)
        or str(idmes_ultima_campanha).strip() in ('', 'NULL', 'None', 'nan', 'NaT')
    ):
        return 1

    try:
        idmes = int(idmes_ultima_campanha)
        ano_atual     = mes_atual // 100
        mes_atual_num = mes_atual % 100
        ano_ultimo    = idmes   // 100
        mes_ultimo    = idmes   %  100
        meses_atras   = (ano_atual - ano_ultimo) * 12 + (mes_atual_num - mes_ultimo)

        if   meses_atras >= 6: return 2
        elif meses_atras >= 3: return 3
        else:                  return 4
    except Exception:
        return 3


def label_tier(t):
    return {
        1: 'Tier 1 – Nunca contactado (NULL)',
        2: 'Tier 2 – ≥ 6 meses sem campanha',
        3: 'Tier 3 – 3 a 5 meses sem campanha',
        4: 'Tier 4 – < 3 meses sem campanha',
    }.get(t, f'Tier {t}')


def label_pressao(p):
    if   p == 0: return '0 – Sem pressão'
    elif p == 1: return '1 campanha'
    elif p == 2: return '2 campanhas'
    elif p == 3: return '3 campanhas'
    else:        return '> 3 campanhas'


def nivel_escassez(e):
    if   e > ESCASSEZ_THRESHOLD_CRITICA: return 'CRÍTICO'
    elif e > ESCASSEZ_THRESHOLD_ALTA:    return 'ALTO'
    elif e > ESCASSEZ_THRESHOLD_MEDIA:   return 'MÉDIO'
    else:                                return 'BAIXO'


def sep(char='=', width=80):
    print(char * width)


def titulo(texto, char='='):
    sep(char)
    print(f'  {texto}')
    sep(char)


print("✓ Funções auxiliares definidas.")


## Secção 1 – Visão Geral do Projecto

In [ ]:
# =============================================================================
# SECÇÃO 1 – VISÃO GERAL DO PROJECTO (PLANO COMPLETO)
# =============================================================================

sep()
print("  1. VISÃO GERAL DO PROJECTO – PLANO COMPLETO")
sep()
print()

total_camps    = len(df_plano)
total_vol      = df_plano['volumetria'].sum()
camps_passadas = df_plano[df_plano['data'].dt.date <  hoje.date()]
camps_hoje     = df_plano[df_plano['data'].dt.date == hoje.date()]
camps_futuras  = df_plano[df_plano['data'].dt.date >  hoje.date()]

print(f"  Data de execução      : {hoje.strftime('%Y-%m-%d')}")
print(f"  Total de campanhas    : {total_camps}")
print(f"    • Passadas          : {len(camps_passadas)}")
print(f"    • Hoje              : {len(camps_hoje)}")
print(f"    • Futuras           : {len(camps_futuras)}")
print(f"  Volume total planeado : {total_vol:,} contactos")
print()

print("  Distribuição por Tipo de Oferta:")
tp = (
    df_plano.groupby('tipo oferta', dropna=False)
    .agg(n_campanhas=('campanha', 'count'), volumetria=('volumetria', 'sum'))
    .sort_values('n_campanhas', ascending=False)
)
for tipo, row in tp.iterrows():
    pct = row['volumetria'] / total_vol * 100 if total_vol else 0
    print(f"    {str(tipo):<30}  {row['n_campanhas']:>3} campanha(s)   {row['volumetria']:>8,} contactos  ({pct:.1f}%)")
print()

print("  Distribuição temporal (por data de envio):")
dist_data = (
    df_plano.groupby(df_plano['data'].dt.date)
    .agg(n=('campanha', 'count'), vol=('volumetria', 'sum'))
    .sort_index()
)
for data, row in dist_data.iterrows():
    marcador = '  ◄ HOJE' if data == hoje.date() else ''
    print(f"    {str(data)}  {row['n']:>3} campanha(s)   {row['vol']:>8,} contactos{marcador}")
print()


## Secção 2 – Campanhas de Hoje

In [ ]:
# =============================================================================
# SECÇÃO 2 – CAMPANHAS A DATA DE HOJE
# =============================================================================

sep()
print("  2. CAMPANHAS A DATA DE HOJE")
sep()
print()

camps_hoje_df = df_plano[df_plano['data'].dt.date == hoje.date()].copy()

if camps_hoje_df.empty:
    print("  ℹ  Não existem campanhas agendadas para hoje.")
else:
    total_vol_hoje = camps_hoje_df['volumetria'].sum()
    print(f"  Total de campanhas hoje : {len(camps_hoje_df)}")
    print(f"  Volume total hoje       : {total_vol_hoje:,} contactos")
    print()

    print(f"  {'Campanha':<35} {'Tipo':<20} {'Volumetria':>12}  Escassez  Nível")
    print(f"  {'-'*35} {'-'*20} {'-'*12}  {'-'*8}  {'-'*8}")

    for _, row in camps_hoje_df.sort_values('volumetria', ascending=False).iterrows():
        camp = row['campanha']
        vol  = int(row['volumetria'])
        tipo = str(row.get('tipo oferta', 'PCD'))

        col_eleg = next((c for c in df_baseenvio.columns if c == camp), None)
        if col_eleg is not None:
            elegiveis = int((df_baseenvio[col_eleg] == 1).sum())
            escassez  = vol / max(elegiveis, 1)
            nivel     = nivel_escassez(escassez)
            esc_str   = f'{escassez:.2f}'
        else:
            nivel   = 'N/D'
            esc_str = 'N/D'

        print(f"  {camp:<35} {tipo:<20} {vol:>12,}  {esc_str:>8}  {nivel}")

print()


## Secção 3 – Escassez das Campanhas

In [ ]:
# =============================================================================
# SECÇÃO 3 – CARACTERIZAÇÃO DE ESCASSEZ DAS CAMPANHAS
# =============================================================================

sep()
print("  3. CARACTERIZAÇÃO DE ESCASSEZ DAS CAMPANHAS")
sep()
print()

resultados = []
for _, row in df_plano.iterrows():
    camp = row['campanha']
    vol  = int(row['volumetria'])
    data = row['data']

    if camp in df_baseenvio.columns:
        elegiveis = int((df_baseenvio[camp] == 1).sum())
    else:
        elegiveis = 0

    escassez = vol / max(elegiveis, 1)
    resultados.append({
        'campanha'  : camp,
        'data'      : data,
        'tipo'      : str(row.get('tipo oferta', '')),
        'volumetria': vol,
        'elegiveis' : elegiveis,
        'escassez'  : escassez,
        'nivel'     : nivel_escassez(escassez),
    })

df_esc = pd.DataFrame(resultados)

contagem = df_esc['nivel'].value_counts().reindex(
    ['CRÍTICO', 'ALTO', 'MÉDIO', 'BAIXO'], fill_value=0
)
escassas     = contagem['CRÍTICO'] + contagem['ALTO']
nao_escassas = contagem['MÉDIO']   + contagem['BAIXO']

print(f"  Threshold escassez crítica : > {ESCASSEZ_THRESHOLD_CRITICA:.0%}")
print(f"  Threshold escassez alta    : > {ESCASSEZ_THRESHOLD_ALTA:.0%}")
print()
print(f"  Campanhas ESCASSAS   (CRÍTICO + ALTO) : {escassas:>4}")
print(f"    • CRÍTICO  (> {ESCASSEZ_THRESHOLD_CRITICA:.0%})              : {contagem['CRÍTICO']:>4}")
print(f"    • ALTO     (> {ESCASSEZ_THRESHOLD_ALTA:.0%})              : {contagem['ALTO']:>4}")
print(f"  Campanhas NÃO ESCASSAS (MÉDIO + BAIXO): {nao_escassas:>4}")
print(f"    • MÉDIO    (> {ESCASSEZ_THRESHOLD_MEDIA:.0%})              : {contagem['MÉDIO']:>4}")
print(f"    • BAIXO    (≤ {ESCASSEZ_THRESHOLD_MEDIA:.0%})              : {contagem['BAIXO']:>4}")
print()

df_detalhe = df_esc.sort_values('escassez', ascending=False)
print(f"  {'Campanha':<35} {'Data':<12} {'Tipo':<20} {'Vol':>8}  {'Elegiveis':>10}  {'Escassez':>9}  Nível")
print(f"  {'-'*35} {'-'*12} {'-'*20} {'-'*8}  {'-'*10}  {'-'*9}  {'-'*8}")
for _, r in df_detalhe.iterrows():
    data_str = r['data'].strftime('%Y-%m-%d') if pd.notna(r['data']) else 'N/D'
    print(
        f"  {r['campanha']:<35} {data_str:<12} {r['tipo']:<20} "
        f"{r['volumetria']:>8,}  {r['elegiveis']:>10,}  {r['escassez']:>9.2f}  {r['nivel']}"
    )
print()

df_esc


## Secção 4 – Caracterização da Base de Clientes por Tier

In [ ]:
# =============================================================================
# SECÇÃO 4 – CARACTERIZAÇÃO DA BASE DE CLIENTES POR TIER
# =============================================================================

sep()
print("  4. CARACTERIZAÇÃO DA BASE DE CLIENTES POR TIER")
sep()
print()

if 'IdMesUltimaCampanha' not in df_baseenvio.columns:
    print("  ⚠  Coluna IdMesUltimaCampanha não encontrada no df_baseenvio.")
else:
    df_tier = df_baseenvio.copy()
    df_tier['tier'] = df_tier['IdMesUltimaCampanha'].apply(
        lambda x: calcular_tier_antiguidade(x, mes_atual)
    )

    total = len(df_tier)
    print(f"  Total de clientes na base : {total:,}")
    print()
    print(f"  {'Tier':<40} {'Clientes':>10}  {'%':>8}")
    print(f"  {'-'*40} {'-'*10}  {'-'*8}")
    for t in [1, 2, 3, 4]:
        n   = int((df_tier['tier'] == t).sum())
        pct = n / total * 100 if total else 0
        print(f"  {label_tier(t):<40} {n:>10,}  {pct:>7.1f}%")
    print()

    display(
        df_tier.groupby('tier')
        .size().rename('clientes')
        .reset_index()
        .assign(
            tier_label=lambda d: d['tier'].map(label_tier),
            pct=lambda d: d['clientes'] / d['clientes'].sum() * 100
        )
        [['tier_label', 'clientes', 'pct']]
        .rename(columns={'tier_label': 'Tier', 'clientes': 'Clientes', 'pct': '%'})
    )


## Secção 5 – Caracterização da Base de Clientes por Pressão Comercial

In [ ]:
# =============================================================================
# SECÇÃO 5 – CARACTERIZAÇÃO DA BASE DE CLIENTES POR PRESSÃO COMERCIAL
# =============================================================================

sep()
print("  5. CARACTERIZAÇÃO DA BASE DE CLIENTES POR PRESSÃO COMERCIAL")
sep()
print()

if 'PressaoComercial' not in df_baseenvio.columns:
    print("  ⚠  Coluna PressaoComercial não encontrada no df_baseenvio.")
else:
    df_pressao = df_baseenvio.copy()
    df_pressao['pressao_num'] = pd.to_numeric(df_pressao['PressaoComercial'], errors='coerce').fillna(0).astype(int)
    df_pressao['pressao_grp'] = df_pressao['pressao_num'].apply(label_pressao)

    ordem = [label_pressao(p) for p in [0, 1, 2, 3, 4]]

    total = len(df_pressao)
    print(f"  Total de clientes na base : {total:,}")
    print()
    print(f"  {'Pressão Comercial':<30} {'Clientes':>10}  {'%':>8}")
    print(f"  {'-'*30} {'-'*10}  {'-'*8}")
    for grp in ordem:
        n   = int((df_pressao['pressao_grp'] == grp).sum())
        pct = n / total * 100 if total else 0
        print(f"  {grp:<30} {n:>10,}  {pct:>7.1f}%")
    print()

    display(
        df_pressao.groupby('pressao_grp')
        .size().rename('clientes')
        .reset_index()
        .assign(pct=lambda d: d['clientes'] / d['clientes'].sum() * 100)
        .rename(columns={'pressao_grp': 'Pressão Comercial', 'clientes': 'Clientes', 'pct': '%'})
    )


## Secção 6 – Análise Cruzada Tier × Pressão Comercial

In [ ]:
# =============================================================================
# SECÇÃO 6 – ANÁLISE CRUZADA: TIER × PRESSÃO COMERCIAL
# =============================================================================

sep()
print("  6. ANÁLISE CRUZADA: TIER × PRESSÃO COMERCIAL")
sep()
print()

colunas_necessarias = {'IdMesUltimaCampanha', 'PressaoComercial'}
if not colunas_necessarias.issubset(df_baseenvio.columns):
    faltam = colunas_necessarias - set(df_baseenvio.columns)
    print(f"  ⚠  Colunas em falta no df_baseenvio: {faltam}")
else:
    df_crz = df_baseenvio.copy()
    df_crz['tier'] = df_crz['IdMesUltimaCampanha'].apply(
        lambda x: calcular_tier_antiguidade(x, mes_atual)
    )
    df_crz['pressao_num'] = pd.to_numeric(df_crz['PressaoComercial'], errors='coerce').fillna(0).astype(int)
    df_crz['pressao_grp'] = df_crz['pressao_num'].apply(
        lambda p: '>3' if p > 3 else str(p)
    )

    cols_pressao = ['0', '1', '2', '3', '>3']
    tiers        = [1, 2, 3, 4]

    tabela = pd.crosstab(df_crz['tier'], df_crz['pressao_grp'])
    for c in cols_pressao:
        if c not in tabela.columns:
            tabela[c] = 0
    tabela = tabela[cols_pressao]
    tabela['TOTAL'] = tabela.sum(axis=1)

    total_geral = tabela['TOTAL'].sum()

    header_pressao = '  '.join(f'{c:>10}' for c in cols_pressao)
    print(f"  {'Tier':<40} {header_pressao}  {'TOTAL':>10}")
    print(f"  {'-'*40} {'  '.join(['-'*10]*5)}  {'-'*10}")

    for t in tiers:
        if t not in tabela.index:
            continue
        row_vals = '  '.join(f"{int(tabela.loc[t, c]):>10,}" for c in cols_pressao)
        total_t  = int(tabela.loc[t, 'TOTAL'])
        pct_t    = total_t / total_geral * 100 if total_geral else 0
        print(f"  {label_tier(t):<40} {row_vals}  {total_t:>10,}  ({pct_t:.1f}%)")

    sep('-')
    total_vals = '  '.join(f"{int(tabela[c].sum()):>10,}" for c in cols_pressao)
    print(f"  {'TOTAL':<40} {total_vals}  {int(total_geral):>10,}")
    print()

    alta_pressao = df_crz[df_crz['pressao_num'] > 3]
    if not alta_pressao.empty:
        print("  Clientes com pressão > 3 campanhas, por Tier:")
        for t in tiers:
            n = int((alta_pressao['tier'] == t).sum())
            if n > 0:
                pct = n / len(alta_pressao) * 100
                print(f"    {label_tier(t):<40} {n:>8,}  ({pct:.1f}%)")
        print()

    # Tabela cruzada como DataFrame para display
    tabela_display = tabela.copy()
    tabela_display.index = [label_tier(t) for t in tabela_display.index]
    display(tabela_display)


## Secção 7 – Elegibilidade e Cobertura

In [ ]:
# =============================================================================
# SECÇÃO 7 – ELEGIBILIDADE E COBERTURA DA BASE DE CLIENTES
# =============================================================================

sep()
print("  7. ELEGIBILIDADE E COBERTURA DA BASE DE CLIENTES")
sep()
print()

colunas_elegibilidade = [c for c in df_baseenvio.columns if c.startswith('iXS_')]
if not colunas_elegibilidade:
    print("  ⚠  Sem colunas de elegibilidade (iXS_*) encontradas no df_baseenvio.")
else:
    total_clientes  = len(df_baseenvio)
    tem_elegivel    = (df_baseenvio[colunas_elegibilidade].sum(axis=1) > 0).sum()
    sem_elegivel    = total_clientes - tem_elegivel
    media_elegiveis = df_baseenvio[colunas_elegibilidade].sum(axis=1).mean()

    print(f"  Total de clientes                   : {total_clientes:>10,}")
    print(f"  Com pelo menos 1 campanha elegível  : {tem_elegivel:>10,}  ({tem_elegivel/total_clientes*100:.1f}%)")
    print(f"  Sem nenhuma campanha elegível       : {sem_elegivel:>10,}  ({sem_elegivel/total_clientes*100:.1f}%)")
    print(f"  Média de campanhas por cliente      : {media_elegiveis:>10.2f}")
    print()

    camps_mes = df_plano[
        df_plano['data'].dt.month == hoje.month
    ]['campanha'].tolist()

    camps_com_dados = [c for c in camps_mes if c in df_baseenvio.columns]
    if camps_com_dados:
        print(f"  Elegibilidade por campanha (mês actual – {hoje.strftime('%Y-%m')}):")
        print(f"  {'Campanha':<35} {'Volumetria':>10}  {'Elegíveis':>10}  {'Cobertura':>10}  Escassez")
        print(f"  {'-'*35} {'-'*10}  {'-'*10}  {'-'*10}  {'-'*8}")

        rows_eleg = []
        for camp in camps_com_dados:
            vol  = int(df_plano.loc[df_plano['campanha'] == camp, 'volumetria'].iloc[0])
            eleg = int((df_baseenvio[camp] == 1).sum())
            cob  = eleg / total_clientes * 100 if total_clientes else 0
            esc  = vol  / max(eleg, 1)
            nivel = nivel_escassez(esc)
            print(
                f"  {camp:<35} {vol:>10,}  {eleg:>10,}  {cob:>9.1f}%  {esc:.2f} {nivel}"
            )
            rows_eleg.append({
                'Campanha': camp, 'Volumetria': vol,
                'Elegíveis': eleg, 'Cobertura (%)': round(cob, 1),
                'Escassez': round(esc, 2), 'Nível': nivel
            })
        print()
        display(pd.DataFrame(rows_eleg))

    print()
    sep()
    print("  FIM DA CARACTERIZAÇÃO")
    sep()
